# Foosball SAC Agent — Kaggle Training v2

Train a **Soft Actor-Critic (SAC)** reinforcement learning agent to play foosball using:
- **MuJoCo** physics simulation (`foosball_sim/v2/`)
- **Stable-Baselines3** SAC implementation
- **Protagonist-Antagonist** self-play curriculum via `SinglePlayerTrainingEngine`

**Repo:** https://github.com/carlkaziboni/Foosball_CU.git  
**Output:** models saved to `/kaggle/working/models/` — downloadable as zip at the end.

---
Run cells **top to bottom**. GPU (P100/T4) auto-detected and hyperparameters scaled accordingly.

## 1. Install System & Python Dependencies

Installs headless OpenGL libraries, MuJoCo, Stable-Baselines3, and all supporting packages.  
The repo is cloned (or pulled if already present) and added to `sys.path`.

In [1]:
import subprocess, sys, os, shutil

def run(cmd):
    subprocess.check_call(cmd)

def pip(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])

# ── System packages for headless OpenGL rendering ────────────────────────────
run(["apt-get", "install", "-y", "-q",
     "xvfb", "libgl1-mesa-glx", "libosmesa6", "libglfw3", "ffmpeg"])

# ── Python packages ────────────────────────────────────────────────────────────
pip(
    "mujoco",
    "stable-baselines3[extra]",
    "gymnasium",
    "shimmy>=0.2.0",
    "pyvirtualdisplay",
    "glfw",
    "tensorboard",
)

# ── Clone / update repo ────────────────────────────────────────────────────────
REPO_URL = "https://github.com/carlkaziboni/Foosball_CU.git"
REPO_DIR = "/kaggle/working/Foosball_CU"

# Always do a fresh shallow clone to guarantee we have the latest commit.
# (git pull on a --depth 1 clone often fails with divergent-branch errors.)
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
print(f"Repo cloned → {REPO_DIR}")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Install any project-level requirements
req_file = os.path.join(REPO_DIR, "requirements.txt")
if os.path.exists(req_file):
    pip("-r", req_file)
    print("requirements.txt installed ✓")

print("\nCell 1 done ✓")

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
Suggested packages:
  libgles1 libvulkan1
The following NEW packages will be installed:
  libglfw3 libosmesa6
0 upgraded, 2 newly installed, 0 to remove and 134 not upgraded.
Need to get 3,198 kB of archives.
After this operation, 12.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libglfw3 amd64 3.3.6-1 [83.2 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libosmesa6 amd64 23.2.1-1ubuntu3.1~22.04.3 [3,115 kB]
Fetched 3,198 kB in 3s (1,082 kB/s)
Selecting previously unselected package libglfw3:amd64.
(Reading database ... 124463 files and directories currently installed.)
Preparing to unpack .../libglfw3_3.3.6-1_amd64.deb ...
Unpa

Cloning into '/kaggle/working/Foosball_CU'...


Repo cloned → /kaggle/working/Foosball_CU

Cell 1 done ✓


Updating files: 100% (360/360), done.


## 2. Headless Rendering Setup & GLFW Patch

Sets `MUJOCO_GL=osmesa`, starts a virtual display, then **patches `MujocoTableRenderMixin` to a no-op class** before `FoosballEnv` is ever imported.  
This prevents GLFW window errors in Kaggle's headless kernel.

In [2]:
import os, sys, warnings

# ── Headless env vars — must be set BEFORE any mujoco/glfw import ─────────────
os.environ["MUJOCO_GL"]         = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"

# ── Virtual display ────────────────────────────────────────────────────────────
from pyvirtualdisplay import Display
_display = Display(visible=False, size=(1280, 960))
_display.start()
os.environ["DISPLAY"] = ":0"
print("Virtual display started ✓")

# ── Silence gym deprecation warnings ──────────────────────────────────────────
warnings.filterwarnings("ignore", message=".*upgrade to Gymnasium.*")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="gym")

# ── Output directories ─────────────────────────────────────────────────────────
REPO_DIR  = "/kaggle/working/Foosball_CU"
MODEL_DIR = "/kaggle/working/models"
LOG_DIR   = "/kaggle/working/logs"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR,   exist_ok=True)

# ── Patch GLFW mixin to no-op BEFORE FoosballEnv is imported ──────────────────
import ai_agents.v2.gym.mujoco_table_render_mixin as _mixin_module

class _HeadlessMixin:
    """No-op render mixin for Kaggle headless environment."""
    def __init__(self):
        self.viewer = None
        self.window = None
    def _initialize_glfw(self): pass
    def render(self, mode="human"): pass
    def close(self): pass

_mixin_module.MujocoTableRenderMixin = _HeadlessMixin
print("MujocoTableRenderMixin → HeadlessMixin ✓")

print(f"\nMUJOCO_GL  : {os.environ['MUJOCO_GL']}")
print(f"MODEL_DIR  : {MODEL_DIR}")
print(f"LOG_DIR    : {LOG_DIR}")
print("Cell 2 done ✓")

Virtual display started ✓
MujocoTableRenderMixin → HeadlessMixin ✓

MUJOCO_GL  : osmesa
MODEL_DIR  : /kaggle/working/models
LOG_DIR    : /kaggle/working/logs
Cell 2 done ✓


## 3. Environment Factory

Imports `FoosballEnv` (safe now that the mixin is patched) and defines the factory used everywhere.  
`observation_space` is **38-dim** continuous; `action_space` is **8-dim** continuous (4 rods × 2: linear + rotation).

In [3]:
# Mixin is already patched — safe to import FoosballEnv now
from stable_baselines3.common.monitor import Monitor
from ai_agents.v2.gym.full_information_protagonist_antagonist_gym import FoosballEnv

# ── Global antagonist slot — set to a loaded SAC model to enable self-play ─────
_current_antagonist = None  # Phase 1: None = solo training


def sac_foosball_env_factory(x=None):
    """Creates a monitored FoosballEnv using the current global antagonist.
    In Phase 1, _current_antagonist is None (solo).
    In Phase 2, _current_antagonist is a frozen SAC model (self-play).
    """
    env = FoosballEnv(antagonist_model=_current_antagonist)
    env = Monitor(env, filename=os.path.join(LOG_DIR, "monitor"))
    return env


# ── Sanity check ───────────────────────────────────────────────────────────────
_test_env = sac_foosball_env_factory()
print("Observation space :", _test_env.observation_space)
print("Action space      :", _test_env.action_space)
_test_env.close()
del _test_env
print("Cell 3 done ✓  (antagonist slot ready)")


2026-03-11 11:00:03.343967: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773226803.522206      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773226803.573047      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773226803.991833      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773226803.991872      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773226803.991875      24 computation_placer.cc:177] computation placer alr

Observation space : Box(-inf, inf, (38,), float32)
Action space      : Box(-10.0, 10.0, (8,), float32)
Cell 3 done ✓  (antagonist slot ready)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.12/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


## 4. GPU Detection & Stable SAC Agent

Detects available hardware (CUDA P100/T4, MPS, or CPU) and defines `StableSACAgent` — a subclass of `SACFoosballAgent` with:
- `net_arch = [512, 512, 256]` (stable; avoids CPU/GPU tensor mismatch from large nets)
- `use_sde = False` (prevents NaN explosions on CUDA kernels)
- All other hyperparameters tuned for fast foosball learning

In [4]:
import torch
from stable_baselines3 import SAC
from ai_agents.common.train.impl.sac_agent import SACFoosballAgent

# ── Detect device ──────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Training device : CUDA — {gpu_name}")
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Training device : Apple Silicon MPS")
else:
    DEVICE = "cpu"
    print("Training device : CPU")

# ── StableSACAgent — subclass avoids recursion errors on notebook re-run ───────
_true_sac_init = SACFoosballAgent.__dict__["__init__"]

class StableSACAgent(SACFoosballAgent):
    """
    SAC agent with a stable [512, 512, 256] network and use_sde=False.
    DEBUG VERSION: Progress bar disabled, verbose=1, logging every episode.
    """

    def __init__(self, id, env=None,
                 log_dir=LOG_DIR,
                 model_dir=MODEL_DIR,
                 policy_kwargs=None):
        if policy_kwargs is None:
            policy_kwargs = dict(net_arch=[512, 512, 256])
        _true_sac_init(self, id=id, env=env,
                       log_dir=log_dir, model_dir=model_dir,
                       policy_kwargs=policy_kwargs)

    # ── Internal SAC factory ───────────────────────────────────────────────────
    def _make_sac(self, tensorboard_log=None):
        kwargs = dict(
            policy="MlpPolicy",
            env=self.env,
            policy_kwargs=self.policy_kwargs,
            device=DEVICE,
            # ── Learning dynamics ─────────────────────────────────────────────
            learning_rate=3e-4,
            buffer_size=100_000,
            learning_starts=500,      
            batch_size=256,
            tau=0.005,
            gamma=0.99,
            train_freq=4,             
            gradient_steps=2,         
            # ── Entropy tuning ────────────────────────────────────────────────
            ent_coef="auto",
            target_entropy="auto",
            # ── Stability ─────────────────────────────────────────────────────
            use_sde=False,
            verbose=1,                # ← CHANGED: Enabled standard SB3 text output
        )
        if tensorboard_log:
            kwargs["tensorboard_log"] = tensorboard_log
        return SAC(**kwargs)

    def initialize_agent(self):
        try:
            self.load()
            print(f"Agent {self.id} — loaded from checkpoint.")
        except Exception:
            print(f"Agent {self.id} — no checkpoint found, initialising new model.")
            self.model = self._make_sac()

    def learn(self, total_timesteps):
        if self.model is None:
            tb_log = os.path.join(LOG_DIR, f"sac_agent_{self.id}")
            self.model = self._make_sac(tensorboard_log=tb_log)

        callback = self.create_callback(self.env)
        print(f"Agent {self.id} — training for {total_timesteps:,} timesteps ...")
        self.model.learn(
            total_timesteps=total_timesteps,
            callback=callback,
            tb_log_name=f"run_{self.id}",
            progress_bar=False,           # ← CHANGED: Disabled tqdm to prevent deadlocks
            log_interval=1,               # ← CHANGED: Log every episode to track exactly where it stops
            reset_num_timesteps=False,    
        )

print("StableSACAgent defined ✓ (DEBUG MODE ENABLED)")
print("Cell 4 done ✓")

Training device : CUDA — Tesla P100-PCIE-16GB
StableSACAgent defined ✓ (DEBUG MODE ENABLED)
Cell 4 done ✓


## 5. Training Configuration

Kaggle **P100/T4 GPU** quota is ~30 hr/week.  
Defaults are calibrated to stay well within quota while still learning meaningful behaviour.

| Setting | GPU | CPU |
|---|---|---|
| `total_epochs` | 50 | 10 |
| `epoch_timesteps` | 10 000 | 2 000 |
| `cycle_timesteps` | 1 000 | 500 |
| **Total timesteps** | **500 000** | **20 000** |

**Self-play curriculum (reward-triggered):**  
The agent trains solo until the mean reward over the last `self_play_reward_window` episodes reaches `self_play_reward_threshold`. At that point it automatically switches to self-play against a frozen checkpoint of itself, refreshed every `self_play_update_interval` epochs.  
If the threshold is never reached, the full run completes in solo mode.

Adjust the variables in the cell below if you want a longer/shorter run or a different trigger threshold.


In [5]:
IS_GPU = DEVICE == "cuda"

# ── Auto-scale based on available hardware ─────────────────────────────────────
if IS_GPU:
    total_epochs    = 50
    epoch_timesteps = 10_000
    cycle_timesteps = 1_000
else:
    total_epochs    = 10
    epoch_timesteps = 2_000
    cycle_timesteps = 500

# ── Override here if you want a custom run ─────────────────────────────────────
# total_epochs    = 20
# epoch_timesteps = 5_000
# cycle_timesteps = 500

total_timesteps = total_epochs * epoch_timesteps

# ── Self-play curriculum (reward-based trigger) ────────────────────────────────
# Phase 1: solo training until mean reward over the last N episodes >= threshold
# Phase 2: self-play against a frozen checkpoint, refreshed every M epochs
self_play_reward_threshold = 500.0   # Switch when mean reward exceeds this
self_play_reward_window    = 20      # Number of recent episodes to average over
self_play_update_interval  = 5       # Refresh frozen opponent every N epochs in Phase 2

print("=" * 52)
print(f"  Device              : {DEVICE.upper()}")
print(f"  Epochs              : {total_epochs}")
print(f"  Timesteps / epoch   : {epoch_timesteps:,}")
print(f"  Cycle timesteps     : {cycle_timesteps:,}")
print(f"  Total timesteps     : {total_timesteps:,}")
print(f"  ── Self-play trigger ──────────────────")
print(f"  Switch threshold    : mean reward ≥ {self_play_reward_threshold}")
print(f"  Reward window       : last {self_play_reward_window} episodes")
print(f"  Antagonist refresh  : every {self_play_update_interval} epochs (Phase 2)")
print(f"  Model output dir    : {MODEL_DIR}")
print(f"  Log output dir      : {LOG_DIR}")
print("=" * 52)
print("Cell 5 done ✓")


  Device              : CUDA
  Epochs              : 50
  Timesteps / epoch   : 10,000
  Cycle timesteps     : 1,000
  Total timesteps     : 500,000
  ── Self-play trigger ──────────────────
  Switch threshold    : mean reward ≥ 500.0
  Reward window       : last 20 episodes
  Antagonist refresh  : every 5 epochs (Phase 2)
  Model output dir    : /kaggle/working/models
  Log output dir      : /kaggle/working/logs
Cell 5 done ✓


## 6. Agent Manager & Training Engine

`GenericAgentManager` owns one `StableSACAgent`.  
`SinglePlayerTrainingEngine` runs one epoch per call, saves checkpoints, and handles model reloading between epochs.

In [6]:
from ai_agents.common.train.impl.generic_agent_manager import GenericAgentManager
from ai_agents.common.train.impl.single_player_training_engine import SinglePlayerTrainingEngine

# ── Agent manager ──────────────────────────────────────────────────────────────
agent_manager = GenericAgentManager(
    num_agents=1,
    environment_generator=sac_foosball_env_factory,
    agent_class=StableSACAgent,
)
agent_manager.initialize_training_agents()
agent_manager.initialize_frozen_best_models()

# NOTE: No engine is created here — Cell 7 creates a fresh SinglePlayerTrainingEngine
# each epoch so that FoosballEnv instances pick up the current _current_antagonist.

print("Agent manager  : 1 StableSACAgent ✓")
print("Cell 6 done ✓")


✓ Agent 0 using NVIDIA GPU (CUDA) for training
Agent 0 — no checkpoint found, initialising new model.
Using cuda device
Wrapping the env in a DummyVecEnv.
✓ Agent 0 using NVIDIA GPU (CUDA) for training
Frozen agent 0 using current training agent (no saved model)
Agent manager  : 1 StableSACAgent ✓
Cell 6 done ✓


## 7. Run Training (Reward-Triggered Self-Play)

**Phase 1 (Solo):** trains with no opponent until the mean episode reward over the last `self_play_reward_window` episodes reaches `self_play_reward_threshold`.  
**Phase 2 (Self-play):** once the threshold is crossed the agent immediately starts competing against a frozen checkpoint of itself. Every `self_play_update_interval` epochs the frozen opponent is refreshed, so difficulty grows alongside the agent.  

If the threshold is never reached, the full run completes in solo mode.


In [7]:
import time, csv
from stable_baselines3 import SAC

FROZEN_CHECKPOINT = os.path.join(MODEL_DIR, "0", "sac", "best_model", "model.zip")
MONITOR_CSV       = os.path.join(LOG_DIR, "monitor.monitor.csv")

def read_recent_mean_reward(csv_path, window=20):
    """Read the last `window` episode rewards from the SB3 Monitor CSV."""
    if not os.path.exists(csv_path):
        return None
    rewards = []
    try:
        with open(csv_path, "r") as f:
            # SB3 Monitor writes a comment line then a header line before data
            reader = csv.DictReader(row for row in f if not row.startswith("#"))
            for row in reader:
                rewards.append(float(row["r"]))
    except Exception:
        return None
    if not rewards:
        return None
    recent = rewards[-window:]
    return sum(recent) / len(recent)

wall_start = time.time()
global _current_antagonist
_current_antagonist = None   # Start solo

in_self_play      = False
self_play_epoch   = 0        # Counts epochs elapsed since entering Phase 2
epochs_completed  = 0

print(f"Starting Foosball SAC training — {total_epochs} epochs total")
print(f"Self-play triggers when mean reward (last {self_play_reward_window} eps) ≥ {self_play_reward_threshold}\n")

for epoch in range(1, total_epochs + 1):

    # ── Decide whether to refresh the antagonist this epoch ───────────────────
    if in_self_play and (self_play_epoch % self_play_update_interval == 0):
        try:
            _current_antagonist = SAC.load(FROZEN_CHECKPOINT, device=DEVICE)
            print(f"  [Epoch {epoch}] Antagonist refreshed from checkpoint ✓")
        except Exception as e:
            print(f"  [Epoch {epoch}] Checkpoint not found ({e}) — keeping current antagonist")

    # ── Rebuild engine each epoch so FoosballEnv picks up current antagonist ──
    ep_engine = SinglePlayerTrainingEngine(
        agent_manager=agent_manager,
        environment_generator=sac_foosball_env_factory,
    )

    ep_engine.train(
        total_epochs=1,
        epoch_timesteps=epoch_timesteps,
        cycle_timesteps=cycle_timesteps,
    )
    epochs_completed += 1
    if in_self_play:
        self_play_epoch += 1

    # ── Check mean reward and decide whether to enter Phase 2 ─────────────────
    mean_r = read_recent_mean_reward(MONITOR_CSV, window=self_play_reward_window)
    phase_label = "SELF-PLAY" if in_self_play else "SOLO"
    r_str = f"{mean_r:.1f}" if mean_r is not None else "n/a"
    print(f"  [Epoch {epoch}/{total_epochs}] phase={phase_label}  mean_reward(last {self_play_reward_window})={r_str}")

    if not in_self_play and mean_r is not None and mean_r >= self_play_reward_threshold:
        in_self_play    = True
        self_play_epoch = 0
        print(f"\n{'='*60}")
        print(f"  *** Reward threshold reached ({mean_r:.1f} ≥ {self_play_reward_threshold}) ***")
        print(f"  Switching to SELF-PLAY at epoch {epoch + 1}")
        print(f"{'='*60}\n")
        # Load first frozen antagonist immediately on transition
        try:
            _current_antagonist = SAC.load(FROZEN_CHECKPOINT, device=DEVICE)
            print(f"  Initial antagonist loaded ✓")
        except Exception as e:
            print(f"  Could not load initial antagonist ({e}) — Phase 2 will use None until saved")

# ── Clean up ───────────────────────────────────────────────────────────────────
_current_antagonist = None

elapsed = time.time() - wall_start
h, rem = divmod(int(elapsed), 3600)
m, s   = divmod(rem, 60)

print("\n" + "=" * 60)
print(f"Training complete ✓  ({h:02d}h {m:02d}m {s:02d}s)")
print(f"Total timesteps  : {total_timesteps:,}")
if in_self_play:
    print(f"Self-play phase  : started at reward threshold {self_play_reward_threshold}")
else:
    print(f"Self-play phase  : never triggered (max reward {r_str} < {self_play_reward_threshold})")
print(f"Models saved to  : {MODEL_DIR}")


Starting Foosball SAC training — 50 epochs total
Self-play triggers when mean reward (last 20 eps) ≥ 500.0


Starting epoch 1/1
Training for 10000 timesteps
Total timesteps so far: 0

✓ Agent 0 using NVIDIA GPU (CUDA) for training
Frozen agent 0 using current training agent (no saved model)
Wrapping the env in a DummyVecEnv.
Agent 0 — training for 10,000 timesteps ...


/usr/local/lib/python3.12/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.12/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


---------------------------------
| replay/            |          |
|    buffer_size     | 1499     |
| rollout/           |          |
|    ep_len_mean     | 1.5e+03  |
|    ep_rew_mean     | 1.6e+03  |
| time/              |          |
|    episodes        | 1        |
|    fps             | 186      |
|    time_elapsed    | 8        |
|    total_timesteps | 1500     |
| train/             |          |
|    actor_loss      | -18.1    |
|    critic_loss     | 6.52     |
|    ent_coef        | 0.864    |
|    ent_coef_loss   | -1.96    |
|    learning_rate   | 0.0003   |
|    n_updates       | 498      |
---------------------------------
---------------------------------
| replay/            |          |
|    buffer_size     | 2799     |
| rollout/           |          |
|    ep_len_mean     | 1.41e+03 |
|    ep_rew_mean     | 1.18e+03 |
| time/              |          |
|    episodes        | 2        |
|    fps             | 164      |
|    time_elapsed    | 17       |
|    total_tim

## 8. Inspect Saved Checkpoints

Lists every file written under `MODEL_DIR` so you can verify checkpoints are present before downloading.

In [8]:
import glob

all_files = sorted(glob.glob(f"{MODEL_DIR}/**/*", recursive=True))
model_files = [f for f in all_files if os.path.isfile(f)]

if model_files:
    print(f"Found {len(model_files)} saved file(s) under {MODEL_DIR}:\n")
    for f in model_files:
        size_kb = os.path.getsize(f) / 1024
        print(f"  {f}  ({size_kb:.1f} KB)")
else:
    print(f"No files found under {MODEL_DIR} — check training ran successfully.")

Found 3 saved file(s) under /kaggle/working/models:

  /kaggle/working/models/0/sac/best_model/best_model.zip  (18028.8 KB)
  /kaggle/working/models/0/sac/best_model/model.zip  (18028.8 KB)
  /kaggle/working/models/0/sac/checkpoint_6k/model.zip  (18028.8 KB)


## 9. Evaluate Trained Agent

Loads the best saved model and runs `NUM_EVAL_EPISODES` deterministic episodes.  
Prints per-episode reward and a summary (mean / min / max).

In [9]:
NUM_EVAL_EPISODES = 10

# Re-initialise frozen models so they pick up the latest saved checkpoint
agent_manager.initialize_frozen_best_models()
protagonist = agent_manager.get_frozen_best_models()[0]

eval_env = sac_foosball_env_factory()
episode_rewards = []

print(f"Evaluating trained agent — {NUM_EVAL_EPISODES} episodes (deterministic)\n")

for ep in range(NUM_EVAL_EPISODES):
    obs, _ = eval_env.reset()
    done = False
    total_reward = 0.0
    steps = 0

    while not done:
        # protagonist is a SACFoosballAgent — its predict() already unwraps (action, state)
        # and returns just the action array directly
        action = protagonist.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        steps += 1
        done = terminated or truncated

    episode_rewards.append(total_reward)
    print(f"  Episode {ep + 1:>2}/{NUM_EVAL_EPISODES}  reward: {total_reward:>8.2f}  steps: {steps}")

eval_env.close()

mean_r = sum(episode_rewards) / len(episode_rewards)
print(f"\n{'─' * 40}")
print(f"  Mean reward : {mean_r:.2f}")
print(f"  Min  reward : {min(episode_rewards):.2f}")
print(f"  Max  reward : {max(episode_rewards):.2f}")
print(f"{'─' * 40}")
print("Cell 9 done ✓")


✓ Agent 0 using NVIDIA GPU (CUDA) for training
Wrapping the env in a DummyVecEnv.
Agent 0 loaded model from /kaggle/working/models/0/sac/best_model/model.zip
Frozen agent 0 loaded from saved model
Evaluating trained agent — 10 episodes (deterministic)

  Episode  1/10  reward:   163.91  steps: 300
  Episode  2/10  reward:    84.52  steps: 300
  Episode  3/10  reward:    75.39  steps: 300
  Episode  4/10  reward:    88.96  steps: 300
  Episode  5/10  reward:   142.53  steps: 300
  Episode  6/10  reward:    65.61  steps: 300
  Episode  7/10  reward:    87.80  steps: 300
  Episode  8/10  reward:   210.62  steps: 300
  Episode  9/10  reward:   204.97  steps: 300
  Episode 10/10  reward:    90.58  steps: 300

────────────────────────────────────────
  Mean reward : 121.49
  Min  reward : 65.61
  Max  reward : 210.62
────────────────────────────────────────
Cell 9 done ✓


## 10. Package & Download Models

Zips `models/` into `/kaggle/working/foosball_sac_models.zip`.  
The archive is visible in the Kaggle **Output** tab — click the file to download.

In [10]:
import shutil, glob

ARCHIVE_PATH = "/kaggle/working/foosball_sac_models"

shutil.make_archive(
    base_name=ARCHIVE_PATH,
    format="zip",
    root_dir="/kaggle/working",
    base_dir="models",
)

zip_path = f"{ARCHIVE_PATH}.zip"
zip_size_mb = os.path.getsize(zip_path) / 1e6

print(f"Archive created  : {zip_path}")
print(f"Archive size     : {zip_size_mb:.2f} MB")
print(f"\nContents:")
for f in sorted(glob.glob(f"{MODEL_DIR}/**/*", recursive=True)):
    if os.path.isfile(f):
        kb = os.path.getsize(f) / 1024
        print(f"  {f}  ({kb:.1f} KB)")

print("\nDownload from the Kaggle Output tab.")
print("Cell 10 done ✓")

Archive created  : /kaggle/working/foosball_sac_models.zip
Archive size     : 38.55 MB

Contents:
  /kaggle/working/models/0/sac/best_model/best_model.zip  (18028.8 KB)
  /kaggle/working/models/0/sac/best_model/model.zip  (18028.8 KB)
  /kaggle/working/models/0/sac/checkpoint_6k/model.zip  (18028.8 KB)

Download from the Kaggle Output tab.
Cell 10 done ✓
